# Config Forensics

Validate a model-like decoder config before trusting derived claims.

This is the full **PyTorch** version of a challenge from [LLM Quest](https://bankoti.github.io/llm-quest). The in-browser challenge grades numpy; here you work with real tensors. Fill in each `TODO`, then run the checks cell.

Runs on the free Colab CPU runtime; switch to a GPU via *Runtime > Change runtime type* if you want to experiment at scale. PyTorch comes preinstalled.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class DecoderConfig:
    width: int
    layers: int
    query_heads: int
    kv_heads: int
    head_dim: int
    intermediate_width: int
    experts: int = 1
    experts_per_token: int = 1

    def validate(self) -> None:
        positive = (
            self.width,
            self.layers,
            self.query_heads,
            self.kv_heads,
            self.head_dim,
            self.intermediate_width,
            self.experts,
            self.experts_per_token,
        )
        if any(value <= 0 for value in positive):
            raise ValueError("all architecture dimensions must be positive")
        if self.query_heads % self.kv_heads:
            raise ValueError("query_heads must be divisible by kv_heads")
        if self.experts_per_token > self.experts:
            raise ValueError("experts_per_token cannot exceed experts")

    @property
    def query_width(self) -> int:
        return self.query_heads * self.head_dim

    @property
    def kv_width(self) -> int:
        return self.kv_heads * self.head_dim

    @property
    def queries_per_kv_head(self) -> int:
        return self.query_heads // self.kv_heads

    def attention_projection_parameters(self) -> int:
        # Q, K, V, and output projections; biases intentionally omitted.
        return self.width * (
            self.query_width + 2 * self.kv_width + self.query_width
        )

    def kv_cache_bytes(self, tokens: int, bytes_per_scalar: int = 2) -> int:
        return 2 * self.layers * tokens * self.kv_width * bytes_per_scalar


config = DecoderConfig(
    width=4096,
    layers=32,
    query_heads=32,
    kv_heads=8,
    head_dim=128,
    intermediate_width=11008,
)
config.validate()

In [ ]:
assert config.query_width == config.width
assert config.queries_per_kv_head == 4
assert config.kv_cache_bytes(tokens=8192) == 1_073_741_824

try:
    DecoderConfig(4096, 32, 30, 8, 128, 11008).validate()
except ValueError as error:
    assert "divisible" in str(error)
else:
    raise AssertionError("an inconsistent GQA configuration was accepted")

print(
    "Architecture lab 05 passed",
    {
        "gqa_group": config.queries_per_kv_head,
        "attention_parameters": config.attention_projection_parameters(),
        "kv_cache_gib_at_8k": config.kv_cache_bytes(8192) / 2**30,
    },
)